# Lab 2: HIP Kernel Programming

---

## Abstract

<p style="text-align:justify;">In this lab, you will develop and execute HIP GPU kernels using 1D, 2D, and 3D grid configurations. You will create a single workspace containing three HIP applications, each illustrating a different dimensional mapping between GPU threads and data structures. You will compile and run each application independently and analyze the output to understand how kernel dimensionality affects GPU execution behavior.</p>

## Objectives

After completing this lab, you will be able to:

- Prepare and organize HIP applications for GPU execution by creating a structured workspace and building multiple HIP source files
- Launch HIP GPU kernels using 1D, 2D, and 3D grid-and-block configurations to map threads to different data structures
- Apply core HIP runtime APIs for host-device memory allocation, data transfer, kernel configuration, execution, and cleanup
- Validate correct GPU execution by interpreting program output for vector, matrix, and volumetric workloads

## Introduction

<p style="text-align:justify;">Understanding how GPU threads map to data and how kernels are dispatched is fundamental to writing correct and efficient HIP applications. GPU execution follows a hierarchical grid and block model, where thousands of threads execute in parallel under the single instruction, multiple threads (SIMT) execution paradigm. Each thread computes its own indices to determine which portion of the data it processes.</p>

### Progressive Learning Path

This lab introduces fundamental HIP kernel execution concepts through a structured, progressive workflow that focuses on kernel dimensionality:

1. **1D Kernel (Vector SAXPY):** Demonstrates basic thread indexing and how linear data structures map onto a one-dimensional grid of GPU threads
2. **2D Kernel (Matrix Addition):** Introduces two-dimensional grids and blocks, showing how row and column indices are computed and how 2D data structures are flattened in memory
3. **3D Kernel (Volume Scaling):** Demonstrates three-dimensional thread indexing and the mapping of GPU threads to voxels in a volume

This workflow closely follows the HIP host-driven execution model, reinforcing host responsibilities such as memory allocation, data transfer, kernel configuration, and cleanup.

## General Flow

```
┌──────────────┐   ┌──────────────┐   ┌──────────────┐   ┌──────────────┐
│   Step 1:    │   │   Step 2:    │   │   Step 3:    │   │   Step 4:    │
│  Readying    │-->│  Executing   │-->│  Executing   │-->│  Executing   │
│     the      │   │  1D Kernel:  │   │  2D Kernel:  │   │  3D Kernel:  │
│ Environment  │   │    Vector    │   │   Matrix     │   │   Volume     │
│      &       │   │    SAXPY     │   │   Addition   │   │   Scaling    │
│  Workspace   │   │              │   │              │   │              │
└──────────────┘   └──────────────┘   └──────────────┘   └──────────────┘
```

---

# Step 1: Readying the Environment and Setting the Workspace

Verify AMD GPU availability and HIP compiler readiness, then create a structured workspace to organize and build the HIP applications.

## 1.1 Verify AMD GPU detection

This command queries the ROCm runtime and reports all detected AMD GPUs. Successful output confirms that the AMD GPU is visible, the kernel driver is loaded, and ROCm is functioning correctly.

In [ ]:
!rocminfo

## 1.2 Verify the HIP compiler installation

This command prints the installed HIP compiler version, confirming that the HIP toolchain is available for compiling GPU applications.

In [ ]:
!hipcc --version

## 1.3 Create the lab workspace

### 1.3.1 Create the lab directory

This creates a dedicated workspace directory to store all source files for the lab.

In [ ]:
!mkdir -p hip-kernel-dimensions-lab

### 1.3.2 Change into the lab directory

This sets the working directory so that all files are created and compiled in one location.

In [ ]:
%cd hip-kernel-dimensions-lab

---

# Step 2: Executing a 1D Kernel - Vector SAXPY

Create, compile, and execute a HIP application using a 1D kernel, where each GPU thread processes a single element of a vector using basic thread indexing.

## 2.1 Create the 1D SAXPY kernel source

This HIP program demonstrates a one-dimensional kernel that performs the SAXPY operation: **Y = a*X + Y**, where each thread processes one vector element.

In [ ]:
%%writefile 01_1d_saxpy.cpp
#include <hip/hip_runtime.h>
#include <iostream>
#include <vector>
#include <cstdlib>

/* -------------------------------------------
   HIP error-checking macro
   ------------------------------------------- */
#define HIP_CHECK(call) \
do { \
    hipError_t err = call; \
    if (err != hipSuccess) { \
        std::cerr << "HIP error: " \
                  << hipGetErrorString(err) \
                  << " at " << __FILE__ << ":" << __LINE__ \
                  << std::endl; \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)

/* -------------------------------------------
   GPU kernel: SAXPY
   ------------------------------------------- */
__global__ void saxpy(int n, float a, float* x, float* y) {
    int gid = blockIdx.x * blockDim.x + threadIdx.x;
    if (gid < n) {
        y[gid] = a * x[gid] + y[gid];
    }
}

/* -------------------------------------------
   Host code
   ------------------------------------------- */
int main() {
    int N = 16;
    float a = 2.0f;

    std::vector<float> x(N, 1.0f);
    std::vector<float> y(N, 2.0f);

    float *dx = nullptr;
    float *dy = nullptr;

    /* Device memory allocation */
    HIP_CHECK(hipMalloc(&dx, N * sizeof(float)));
    HIP_CHECK(hipMalloc(&dy, N * sizeof(float)));

    /* Host to device copies */
    HIP_CHECK(hipMemcpy(dx, x.data(),
                        N * sizeof(float),
                        hipMemcpyHostToDevice));
    HIP_CHECK(hipMemcpy(dy, y.data(),
                        N * sizeof(float),
                        hipMemcpyHostToDevice));

    /* Kernel launch */
    hipLaunchKernelGGL(
        saxpy,
        dim3((N + 255) / 256),
        dim3(256),
        0, 0,
        N, a, dx, dy
    );

    /* Ensure kernel execution completed */
    HIP_CHECK(hipDeviceSynchronize());

    /* Device to host copy */
    HIP_CHECK(hipMemcpy(y.data(), dy,
                        N * sizeof(float),
                        hipMemcpyDeviceToHost));

    /* Output validation */
    std::cout << "1D SAXPY Output:\n";
    for (float v : y) {
        std::cout << v << " ";
    }
    std::cout << "\n";

    /* Cleanup */
    HIP_CHECK(hipFree(dx));
    HIP_CHECK(hipFree(dy));

    return 0;
}

## 2.2 Understanding the 1D SAXPY Kernel

### Kernel Signature and 1D Thread Indexing

```cpp
__global__ void saxpy(int n, float a, float* x, float* y) {
    int gid = blockIdx.x * blockDim.x + threadIdx.x;
    if (gid < n) {
        y[gid] = a * x[gid] + y[gid];
    }
}
```

<p style="text-align:justify;">In the 1D kernel, threads are indexed using a single global index computed from the x dimension of the grid and block. The formula `gid = blockIdx.x * blockDim.x + threadIdx.x` is fundamental to GPU programming - it allows thousands of threads to work simultaneously on different data elements without conflicts. The `blockIdx.x` identifies which block the thread belongs to, `blockDim.x` tells how many threads are in each block, and `threadIdx.x` is the thread's position within its block. Together, these three values uniquely identify every thread in the grid. Each GPU thread computes one unique index value using this formula, and this global index determines which element of the vector the thread processes. The bounds check `if (gid < n)` is essential because the total number of threads launched might exceed the actual data size due to block size alignment requirements - GPU kernels are launched in blocks of fixed size (here 256 threads), so when the data size isn't a perfect multiple of the block size, some threads at the end won't have valid data to process. Because the data is one-dimensional (a simple vector), no additional index flattening is required - the global index directly maps to the array position. This kernel establishes the baseline execution model used throughout the lab and serves as the reference point for extending thread indexing to higher dimensions.</p>

### 1D Kernel Launch Configuration

```cpp
dim3 threads(256);
dim3 blocks((N + 255) / 256);
hipLaunchKernelGGL(saxpy, blocks, threads, 0, 0, N, a, dx, dy);
```

<p style="text-align:justify;">The kernel is launched using a one-dimensional grid and one-dimensional block configuration, which directly matches the linear structure of the data being processed. Each block contains 256 threads - this is a common choice because it's typically a multiple of the warp/wavefront size on modern GPUs and provides good occupancy without wasting resources. The grid size is computed using the formula `(N + 255) / 256`, which performs ceiling division to ensure that enough blocks are launched to cover all elements in the vector even when N is not evenly divisible by 256. For example, if N is 1000, we need `(1000 + 255) / 256 = 4` blocks, giving us 1024 total threads (4 blocks × 256 threads), where the last 24 threads will be filtered out by the bounds check in the kernel. This launch configuration provides a simple introduction to kernel execution in HIP and demonstrates the fundamental principle that the dimensionality and size of the thread grid should match the shape and size of the data being processed, rather than defaulting to arbitrary configurations.</p>

### Flat Output Validation

```cpp
for (float v : y) {
    std::cout << v << " ";
}
```

<p style="text-align:justify;">The output is printed as a flat list of values, reflecting the one-dimensional nature of the data structure. Since each thread performs the same SAXPY operation `y = a*x + y` on a different element (with x initialized to 1.0, y initialized to 2.0, and a set to 2.0), all output values are identical: `2.0 * 1.0 + 2.0 = 4.0`. This uniform result across all elements makes it easy to verify correctness at a glance - if any value differs, you know immediately that something went wrong with the kernel execution, memory transfers, or initialization. The flat output format makes it straightforward to visually confirm that each GPU thread processed exactly one vector element and that the entire computation completed successfully without errors.</p>

## 2.3 Compile the 1D SAXPY kernel

This compiles the HIP source file into an executable named `saxpy`, targeting the AMD GPU.

In [ ]:
!hipcc 01_1d_saxpy.cpp -o saxpy

## 2.4 Execute the 1D SAXPY kernel

This launches the program, which allocates GPU memory, executes the kernel, and prints the results.

In [ ]:
!./saxpy

**Expected Output:**
```
1D SAXPY Output:
4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
```

This output confirms that the HIP kernel executed correctly, with each GPU thread processing one element of the vector. The input vectors were initialized on the host (x with all 1.0 values, y with all 2.0 values) and copied to device memory before kernel execution. Each thread computed a unique global index and applied the SAXPY operation `y = 2.0 * x + y`, resulting in `2.0 * 1.0 + 2.0 = 4.0` for every element. Since all input values were identical, every element produced the same result (4). The flat output format reflects the one-dimensional data layout and verifies correct thread indexing, memory transfers, and kernel execution.

---

# Step 3: Executing a 2D Kernel - Matrix Addition

Extend the 1D execution model to a 2D kernel by launching a two-dimensional grid of threads, mapping each thread to a specific matrix row and column.

## 3.1 Create the 2D matrix addition kernel source

This HIP program demonstrates a two-dimensional kernel that performs matrix addition, where each thread processes one matrix element using row and column indexing.

In [ ]:
%%writefile 02_2d_matrix_add.cpp
#include <hip/hip_runtime.h>
#include <iostream>
#include <vector>
#include <cstdlib>

/* -------------------------------------------
   HIP error-checking macro
   ------------------------------------------- */
#define HIP_CHECK(call) \
do { \
    hipError_t err = call; \
    if (err != hipSuccess) { \
        std::cerr << "HIP error: " \
                  << hipGetErrorString(err) \
                  << " at " << __FILE__ << ":" << __LINE__ \
                  << std::endl; \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)

/* -------------------------------------------
   GPU kernel: 2D Matrix Addition
   ------------------------------------------- */
__global__ void mat_add(float* A, float* B, float* C, int N) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < N && col < N) {
        int idx = row * N + col;
        C[idx] = A[idx] + B[idx];
    }
}

/* -------------------------------------------
   Host code
   ------------------------------------------- */
int main() {
    int N = 4;

    std::vector<float> A(N * N, 1.0f);
    std::vector<float> B(N * N, 2.0f);
    std::vector<float> C(N * N);

    float *dA = nullptr;
    float *dB = nullptr;
    float *dC = nullptr;

    /* Device memory allocation */
    HIP_CHECK(hipMalloc(&dA, N * N * sizeof(float)));
    HIP_CHECK(hipMalloc(&dB, N * N * sizeof(float)));
    HIP_CHECK(hipMalloc(&dC, N * N * sizeof(float)));

    /* Host to device copies */
    HIP_CHECK(hipMemcpy(dA, A.data(),
                        N * N * sizeof(float),
                        hipMemcpyHostToDevice));
    HIP_CHECK(hipMemcpy(dB, B.data(),
                        N * N * sizeof(float),
                        hipMemcpyHostToDevice));

    /* Kernel launch configuration */
    dim3 threads(16, 16);
    dim3 blocks((N + 15) / 16, (N + 15) / 16);

    hipLaunchKernelGGL(
        mat_add,
        blocks,
        threads,
        0, 0,
        dA, dB, dC, N
    );

    /* Ensure kernel execution is complete */
    HIP_CHECK(hipDeviceSynchronize());

    /* Device to host copy */
    HIP_CHECK(hipMemcpy(C.data(), dC,
                        N * N * sizeof(float),
                        hipMemcpyDeviceToHost));

    /* Output validation */
    std::cout << "2D Matrix Output:\n";
    for (int i = 0; i < N * N; i++) {
        std::cout << C[i] << (((i + 1) % N) == 0 ? "\n" : " ");
    }

    /* Cleanup */
    HIP_CHECK(hipFree(dA));
    HIP_CHECK(hipFree(dB));
    HIP_CHECK(hipFree(dC));

    return 0;
}

## 3.2 Understanding the 2D Matrix Addition Kernel

### Kernel Signature and 2D Thread Indexing

```cpp
__global__ void mat_add(float* A, float* B, float* C, int N) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < N && col < N) {
        int idx = row * N + col;
        C[idx] = A[idx] + B[idx];
    }
}
```

<p style="text-align:justify;">In the 2D kernel, threads are indexed using two coordinates instead of one, extending the indexing pattern from the 1D case. Each thread now computes both a row index and a column index, using the x and y dimensions of the grid and block structures. The column index is computed identically to the 1D case: `col = blockIdx.x * blockDim.x + threadIdx.x`, mapping threads in the x dimension to matrix columns. The row index introduces the y dimension: `row = blockIdx.y * blockDim.y + threadIdx.y`, mapping threads in the y dimension to matrix rows. This two-dimensional indexing allows the kernel to naturally map each GPU thread to a specific matrix element at position (row, col). However, although the matrix is conceptually two-dimensional, GPU memory is fundamentally linear - there is no such thing as a 2D array in device memory. Therefore, the (row, col) location must be converted into a single linear index using row-major ordering: `idx = row * N + col`. In row-major layout, elements are stored row by row in memory, so element (row, col) is located at position `row * N + col` in the flattened array, where N is the number of columns per row. This demonstrates a critical concept: multi-dimensional data structures must ultimately be accessed through linear memory indices on the GPU, and understanding the flattening formula is essential for correct memory access patterns.</p>

### 2D Kernel Launch Configuration

```cpp
dim3 threads(16, 16);
dim3 blocks((N + 15) / 16, (N + 15) / 16);
hipLaunchKernelGGL(mat_add, blocks, threads, 0, 0, dA, dB, dC, N);
```

<p style="text-align:justify;">Unlike the 1D kernel which used only the x dimension, the launch configuration now uses both x and y dimensions for blocks and grids, creating a two-dimensional thread organization that matches the matrix structure. Each block contains a 16×16 tile of threads, meaning 256 threads per block organized in a square pattern - this tile size is chosen because it's a common balance between occupancy and memory access patterns for 2D workloads. The grid dimensions are calculated to cover the entire matrix using the same ceiling division approach as the 1D case, but now applied in both dimensions: `(N + 15) / 16` computes how many 16-thread blocks are needed in each direction. For a 4×4 matrix, this gives `(4 + 15) / 16 = 1` block in both x and y, so we launch a 1×1 grid of blocks, where each block contains 16×16 = 256 threads, but only the first 4×4 = 16 threads will actually process data while the remaining threads are filtered out by the bounds check. This reinforces a fundamental principle: kernel launch dimensions should match the shape of the data being processed. For 2D data like matrices, we use 2D blocks and grids; for 1D data like vectors, we use 1D configurations. This natural mapping makes the code more intuitive to read and often leads to better memory access patterns on the GPU.</p>

### Structured Output Validation

```cpp
for (int i = 0; i < N * N; ++i) {
    std::cout << C[i] << (((i + 1) % N) == 0 ? "\n" : " ");
}
```

<p style="text-align:justify;">Instead of printing a flat list of values as in the 1D kernel, the output is printed row by row to reflect the two-dimensional structure of the matrix. The expression `(((i + 1) % N) == 0 ? "\n" : " ")` adds a newline after every N elements, effectively breaking the linear output into rows that match the conceptual 2D layout of the data. This visual structure makes it much easier to validate that each matrix element was processed correctly - you can see the matrix shape in the output and verify that the dimensions are correct. Since matrix A was initialized with all values 1.0 and matrix B with all values 2.0, every element in the result matrix C should be 3.0. The row-by-row output makes this validation intuitive and demonstrates how to present multi-dimensional results in a human-readable format.</p>

## 3.3 Compile the 2D matrix addition kernel

This builds the executable for the 2D matrix kernel.

In [ ]:
!hipcc 02_2d_matrix_add.cpp -o matrix_add

## 3.4 Execute the 2D matrix addition kernel

This executes the 2D kernel and prints the resulting matrix.

In [ ]:
!./matrix_add

**Expected Output:**
```
2D Matrix Output:
3 3 3 3
3 3 3 3
3 3 3 3
3 3 3 3
```

<p style="text-align:justify;">This output shows that the 2D matrix addition kernel executed correctly. Each GPU thread computed one matrix element using its row and column indices and performed the operation C = A + B. Since matrix A was initialized with all values 1 and matrix B with all values 2, every element in the result matrix is 3. Printing the output row by row reflects the two-dimensional structure of the data and confirms correct thread indexing, memory access with proper index flattening, and kernel execution.</p>

---

# Step 4: Executing a 3D Kernel - Volume Scaling

Execute a HIP application using a 3D kernel, where each GPU thread operates on one voxel in a three-dimensional volume using x, y, and z indexing.

## 4.1 Create the 3D volume scaling kernel source

This HIP program demonstrates a three-dimensional kernel that scales volumetric data, where each thread processes one voxel using x, y, and z coordinates.

In [ ]:
%%writefile 03_3d_volume_scale.cpp
#include <hip/hip_runtime.h>
#include <iostream>
#include <vector>
#include <cstdlib>

/* -------------------------------------------
   HIP error-checking macro
   ------------------------------------------- */
#define HIP_CHECK(call) \
do { \
    hipError_t err = call; \
    if (err != hipSuccess) { \
        std::cerr << "HIP error: " \
                  << hipGetErrorString(err) \
                  << " at " << __FILE__ << ":" << __LINE__ \
                  << std::endl; \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)

/* -------------------------------------------
   GPU kernel: 3D Volume Scaling
   ------------------------------------------- */
__global__ void scale(float* v, int X, int Y, int Z) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    if (x < X && y < Y && z < Z) {
        int idx = z * (X * Y) + y * X + x;
        v[idx] *= 10.0f;
    }
}

/* -------------------------------------------
   Host code
   ------------------------------------------- */
int main() {
    int X = 4, Y = 4, Z = 4;
    int N = X * Y * Z;

    std::vector<float> v(N, 1.0f);
    float* dv = nullptr;

    /* Device memory allocation */
    HIP_CHECK(hipMalloc(&dv, N * sizeof(float)));

    /* Host to device copy */
    HIP_CHECK(hipMemcpy(dv, v.data(),
                        N * sizeof(float),
                        hipMemcpyHostToDevice));

    /* Kernel launch configuration */
    dim3 threads(4, 4, 4);

    hipLaunchKernelGGL(
        scale,
        dim3(1),
        threads,
        0, 0,
        dv, X, Y, Z
    );

    /* Ensure kernel execution is complete */
    HIP_CHECK(hipDeviceSynchronize());

    /* Device to host copy */
    HIP_CHECK(hipMemcpy(v.data(), dv,
                        N * sizeof(float),
                        hipMemcpyDeviceToHost));

    /* Output validation (sample values) */
    std::cout << "3D Volume Samples:\n";
    std::cout << v[0] << " "
              << v[N / 2] << " "
              << v[N - 1] << "\n";

    /* Cleanup */
    HIP_CHECK(hipFree(dv));

    return 0;
}

## 4.2 Understanding the 3D Volume Scaling Kernel

### Kernel Signature and 3D Thread Indexing

```cpp
__global__ void scale(float* v, int X, int Y, int Z) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    if (x < X && y < Y && z < Z) {
        int idx = z * (X * Y) + y * X + x;
        v[idx] *= 10.0f;
    }
}
```

<p style="text-align:justify;">The 3D kernel introduces three-dimensional thread indexing, extending the indexing pattern from 2D to fully utilize all three dimensions available in the CUDA/HIP execution model. Each thread now computes three coordinates (x, y, z) instead of two, using the familiar pattern applied to each dimension: the x coordinate is computed from blockIdx.x, blockDim.x, and threadIdx.x; the y coordinate from the y components; and the z coordinate from the z components. This allows the kernel to naturally map each GPU thread to a specific voxel (volumetric pixel) inside a three-dimensional volume, identified by its (x, y, z) position. However, just as matrices must be flattened to 1D for memory storage, volumes must also be stored linearly in GPU memory, requiring a more complex flattening formula. The linear index for a 3D volume uses the formula `idx = z * (X * Y) + y * X + x`, which can be understood by considering how 3D data is laid out: we can think of a volume as a stack of 2D slices arranged along the z axis. Each z slice is an entire X×Y plane containing X*Y elements. So to reach slice z, we skip past z complete planes, giving us the offset `z * (X * Y)`. Within that slice, we then navigate to row y, which requires skipping y complete rows of X elements each, adding `y * X` to our offset. Finally, we add x to reach the specific column within that row. This demonstrates how higher-dimensional data structures are systematically mapped into linear memory layouts for GPU access, building on the row-major flattening concept from 2D but extended to three dimensions.</p>

### 3D Kernel Launch Configuration

```cpp
dim3 threads(4, 4, 4);
hipLaunchKernelGGL(scale, dim3(1), threads, 0, 0, dv, X, Y, Z);
```

<p style="text-align:justify;">The kernel launch now uses a three-dimensional block configuration, where each block contains threads organized in x, y, and z dimensions, creating a cube of threads rather than a 2D tile or 1D line. The configuration `dim3(4, 4, 4)` creates blocks containing 4×4×4 = 64 threads arranged in a cubic pattern. Since our volume is exactly 4×4×4 voxels, a single block with 64 threads perfectly covers the entire dataset, which is why the grid size is `dim3(1)` - we only need one block. This simplified configuration (one block containing the exact number of threads needed) makes the mapping between threads and voxels completely explicit and easier to understand conceptually. In a real-world application processing larger volumes, you would calculate the grid dimensions in all three directions using ceiling division similar to the 2D case, but for learning purposes this 1:1 mapping between threads and voxels eliminates complexity and makes the indexing logic transparent. This emphasizes that blocks themselves can be multi-dimensional structures, not just grids - you can have a 3D grid of 3D blocks, giving you full flexibility in how you organize threads to match your data.</p>

### Sample-Based Output Validation

```cpp
std::cout << v[0] << " " << v[N / 2] << " " << v[N - 1] << "\n";
```

<p style="text-align:justify;">Unlike 1D and 2D workloads where we printed complete or structured output, printing the entire 3D dataset is impractical because a 4×4×4 volume contains 64 values, and larger volumes would contain thousands or millions of voxels. Instead, this kernel introduces a realistic GPU debugging technique: sampling representative values to verify correctness without overwhelming the output. The code prints three strategically chosen values: `v[0]` (the first element at position (0,0,0)), `v[N/2]` (the middle element somewhere near the center of the volume), and `v[N-1]` (the last element at position (3,3,3)). This sampling strategy allows you to quickly verify that the kernel executed correctly across the entire volume - if these widely-separated elements all show the expected result (10.0, since each voxel was initialized to 1.0 and multiplied by 10), it's highly likely that all voxels were processed correctly. This technique is commonly used when debugging GPU kernels that process large datasets, where examining every output value would be impractical. The ability to intelligently sample and validate results is an important skill for GPU programming, especially as you move beyond tutorial-sized datasets to production workloads with millions or billions of elements.</p>

## 4.3 Compile the 3D volume scaling kernel

This builds the executable for the 3D volume kernel.

In [ ]:
!hipcc 03_3d_volume_scale.cpp -o volume_scale

## 4.4 Execute the 3D volume scaling kernel

This runs the 3D kernel and prints representative voxel values.

In [ ]:
!./volume_scale

**Expected Output:**
```
3D Volume Samples:
10 10 10
```

<p style="text-align:justify;">This output confirms that the 3D volume scaling kernel executed correctly. Each GPU thread processed one voxel in the volume and multiplied its value by 10. The printed values represent the first, middle, and last elements of the volume, all of which show 10 (original value 1.0 × 10 = 10.0), indicating that the scaling operation was applied uniformly across the entire 3D dataset. This validates that the three-dimensional indexing worked correctly, the index flattening formula properly mapped (x,y,z) coordinates to linear memory positions, and kernel execution completed successfully.</p>

---

# Summary

In this lab, you created and executed three HIP applications demonstrating 1D, 2D, and 3D GPU kernels:

✅ **1D Kernel (Vector SAXPY):** Established baseline thread indexing with linear data structures

✅ **2D Kernel (Matrix Addition):** Extended indexing to two dimensions with row-major memory flattening

✅ **3D Kernel (Volume Scaling):** Demonstrated three-dimensional thread organization with volumetric data

## Key Takeaways

You now understand:

- **Thread-to-Data Mapping:** How GPU threads map to data elements using dimension-specific indexing formulas
- **Memory Flattening:** How multi-dimensional data structures are stored linearly in GPU memory
- **Launch Configurations:** How grid and block dimensions should match the structure of your data
- **Progressive Complexity:** How indexing patterns extend from 1D → 2D → 3D while maintaining the same fundamental principles
- **Validation Techniques:** Different output strategies for different dimensional workloads (flat, structured, sampled)

This foundation prepares you for more advanced HIP programming topics including performance optimization, memory access patterns, and debugging techniques on AMD GPUs.